# Experiment 2 — Instruction-Based Image Editing

## 1. Introduction

This experiment compares two generative AI models for instruction-based image editing in a proposed Adobe Express workflow.

**System A:** Qwen-Image-Edit-2511  
**System B:** FLUX.2 Klein 4B

Both models will receive identical source images and editing instructions. Their outputs will be evaluated for instruction adherence, preservation of original content, visual quality, and computational efficiency.

## 2. Model Acquisition

Apple Silicon-optimised model checkpoints are downloaded from Hugging Face using the MLX format.

- Qwen-Image-Edit-2511: Mixed 4-bit/8-bit quantization.
- FLUX.2 Klein 4B: 8-bit quantization.

The models are cached locally to avoid repeated downloads. Differences in quantization will be documented as an experimental limitation.


In [1]:
import os

# Avoid the Xet download mechanism that previously caused errors.
os.environ["HF_HUB_DISABLE_XET"] = "1"

from huggingface_hub import snapshot_download

models = {
    "FLUX.2 Klein 4B": "AbstractFramework/flux.2-klein-4b-8bit",
    "Qwen-Image-Edit-2511": "AbstractFramework/qwen-image-edit-2511-4bit"
}

for name, model_id in models.items():
    print(f"\nDownloading {name}...")

    try:
        snapshot_download(repo_id=model_id, max_workers=2)
        print(f"{name}: Download complete")
    except Exception as e:
        print(f"{name}: Download interrupted — {e}")


/opt/anaconda3/envs/comp8410-adobe/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Fetching 14 files: 100%|██████████| 14/14 [00:00<00:00, 2029.53it/s]


FLUX.2 Klein 4B: Download complete



Fetching 19 files: 100%|██████████| 19/19 [00:00<00:00, 1939.02it/s]

Qwen-Image-Edit-2511: Download complete


## 3. Local Inference Setup

Both models use Apple Silicon-optimised MLX checkpoints. The MLX-Gen inference framework will be used to perform instruction-based image editing locally.

Model outputs will be displayed in the notebook for visual comparison. The experimental settings, including prompts, random seeds, and processing times, will be recorded for reproducibility.

In [2]:

# Install the MLX-Gen runtime for Apple Silicon

%pip install -q -U mlx-gen

from importlib.metadata import version

print("MLX-Gen version:", version("mlx-gen"))
print("MLX-Gen installation complete.")


Note: you may need to restart the kernel to use updated packages.
MLX-Gen version: 0.38.0
MLX-Gen installation complete.


## 4. Evaluation Data

This experiment uses two types of evaluation data.

First, a small benchmark subset is created from the MagicBrush dataset, which is a manually annotated dataset for instruction-guided image editing. A small subset is used because local image editing is computationally expensive and the assignment focuses on evaluation rather than large-scale benchmarking.

Second, a small custom Adobe Express-style test set will be created later. This will allow the models to be compared in a business context more closely aligned with Adobe Express content creation tasks.

For reproducibility, a fixed random seed is used when sampling benchmark examples.

In [ ]:
# Install dataset tools for loading MagicBrush

%pip install -q datasets pandas matplotlib pillow

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from datasets import load_dataset

Note: you may need to restart the kernel to use updated packages.


In [ ]:
# Load the MagicBrush development split

magicbrush_dev = load_dataset("osunlp/MagicBrush", split="dev")

print("MagicBrush dev samples:", len(magicbrush_dev))
print("Columns:", magicbrush_dev.column_names)

In [ ]:
# Keeping only first-turn edits for a simpler and fairer evaluation

magicbrush_single = magicbrush_dev.filter(
    lambda example: example["turn_index"] == 1
)

print("Single-turn samples:", len(magicbrush_single))

In [ ]:
# Create a small reproducible benchmark subset

rng = np.random.default_rng(42)
sample_size = 8

sample_indices = sorted(
    rng.choice(len(magicbrush_single), size=sample_size, replace=False).tolist()
)

editing_benchmark = magicbrush_single.select(sample_indices)

benchmark_summary = pd.DataFrame({
    "Row": list(range(len(editing_benchmark))),
    "img_id": [editing_benchmark[i]["img_id"] for i in range(len(editing_benchmark))],
    "instruction": [editing_benchmark[i]["instruction"] for i in range(len(editing_benchmark))]
})

display(benchmark_summary)

In [ ]:
# Preview the benchmark examples

num_examples = len(editing_benchmark)
fig, axes = plt.subplots(num_examples, 2, figsize=(8, 4 * num_examples))

if num_examples == 1:
    axes = np.array([axes])

for i in range(num_examples):
    source_img = editing_benchmark[i]["source_img"]
    target_img = editing_benchmark[i]["target_img"]
    instruction = editing_benchmark[i]["instruction"]

    axes[i, 0].imshow(source_img)
    axes[i, 0].set_title(f"Source {i}")
    axes[i, 0].axis("off")

    axes[i, 1].imshow(target_img)
    axes[i, 1].set_title(f"Target {i}")
    axes[i, 1].axis("off")

    print(f"Example {i}: {instruction}")

plt.tight_layout()
plt.show()